In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# cdamien@ 2026

# GCP Knowledge Catalog & PySpark Ingestion Pipeline Demo

This notebook showcases how to build a PySpark ingestion pipeline that reads a JSON dataset from Google Cloud Storage (GCS), transforms it by combining the first and last name, and ingests it into BigQuery. It also registers the GCS fileset in **GCP Konwledge Catalog** as a custom entry for metadata and lineage tracking.

## Architecture Overview
```mermaid
graph LR
    GCS[(JSON File)] --> |Registered in| Kowledge Catalog(Custom Entry)
    GCS --> Read| Spark
    Spark --> |Transform: Full Name| Spark
    Spark --> |Ingest via Connector| BQ[(BigQuery Table)]
```

### Steps covered:
1. **Setup & Authenticate**: Sign in to GCP and install dependencies.
2. **Configuration**: Define GCP Project, Bucket, Dataset, and Table names.
3. **Data Generation**: Create mock JSON fileset and upload to GCS.
4. **Data Catalog Registration**: Register GCS fileset in Data Catalog.
5. **BigQuery Initialization**: Set up target Dataset and Table.
6. **PySpark Job**: Run Spark ETL to read a file with the records in JSON format, transform it, and write to BigQuery.

In [2]:
# 1. Install dependencies
%pip install --quiet pyspark==3.5.4 google-cloud-datacatalog google-cloud-storage google-cloud-bigquery google-cloud-datacatalog-lineage

# 2. Authenticate GCP user (if running in Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Google Colab successfully authenticated!")
except ImportError:
    print("Running outside Google Colab. Make sure your local terminal environment is authenticated (e.g., via gcloud auth application-default login).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.6/375.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.
Google Colab successfully authenticated!


In [3]:
# Configure GCP parameters (Change these values to match your GCP project)
PROJECT_ID = "my-project"             # @param {type:"string"}
REGION = "us-central1"             # @param {type:"string"}
BUCKET_NAME = "my-lineage-bucket" # @param {type:"string"}
DATASET_ID = "lineage_dataset"     # @param {type:"string"}
TABLE_ID = "users_ingested_table"   # @param {type:"string"}

# Data Catalog specific entry configuration
ENTRY_GROUP_ID = "spark_demo_group" # @param {type:"string"}
ENTRY_ID = "gcs_users_json"         # @param {type:"string"}

print(f"Configured GCP variables:")
print(f" - Project: {PROJECT_ID}")
print(f" - Region:  {REGION}")
print(f" - Bucket:  gs://{BUCKET_NAME}")
print(f" - Dataset: {DATASET_ID}")
print(f" - Table:   {TABLE_ID}")

Configured GCP variables:
 - Project: my-project
 - Region:  us-central1
 - Bucket:  gs://my-project-lineage-bucket
 - Dataset: lineage_dataset
 - Table:   users_ingested_table


In [ ]:
import json
import os
from google.cloud import storage

# 1. Generate JSON file for our test
os.makedirs("data", exist_ok=True)
mock_data = [
  {"firstname": "Luffy", "lastname": "Monkey D.", "ipaddress": "10.0.0.1", "address": "Foosha Village"},
  {"firstname": "Zoro", "lastname": "Roronoa", "ipaddress": "10.0.0.2", "address": "Shimotsuki Village"},
  {"firstname": "Nami", "lastname": "Cat Burglar", "ipaddress": "10.0.0.3", "address": "Cocoyasi Village"},
  {"firstname": "Usopp", "lastname": "Sogeking", "ipaddress": "10.0.0.4", "address": "Syrup Village"},
  {"firstname": "Sanji", "lastname": "Vinsmoke", "ipaddress": "10.0.0.5", "address": "Baratie"}
]

local_json_path = "data/users.json"
with open(local_json_path, "w") as f:
    for item in mock_data:
        f.write(json.dumps(item) + "\n")

# 2. Upload file to GCS
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(BUCKET_NAME)
    print(f"GCS Bucket gs://{BUCKET_NAME} exists.")
except Exception: #create if not existing
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Successfully created GCS Bucket: gs://{BUCKET_NAME}")

blob = bucket.blob("input/users.json")
blob.upload_from_filename(local_json_path)
print(f"Successfully uploaded data/users.json to gs://{BUCKET_NAME}/input/users.json")

Generated local NDJSON file: data/users.json
GCS Bucket exists.
Successfully uploaded data/users.json 


In [ ]:
# Registering in Knowledge catalog the gcs fileset (in this case the JSON file)
from google.cloud import dataplex_v1

def register_gcs_fileset(project_id, location, entry_group_id, entry_id, gcs_path, display_name):
    client = dataplex_v1.CatalogServiceClient()

    # Ensure Entry Group exists
    entry_group_path = client.entry_group_path(project_id, location, entry_group_id)
    try:
        client.get_entry_group(name=entry_group_path)
        print(f"Entry Group '{entry_group_id}' already exists.")
    except Exception:
        entry_group = dataplex_v1.EntryGroup()
        entry_group.display_name = "Spark Lineage Demo Group"
        entry_group.description = "Entry group for Spark Lineage Demo assets"
        operation = client.create_entry_group(
            parent=client.common_location_path(project_id, location),
            entry_group_id=entry_group_id,
            entry_group=entry_group,
        )
        operation.result()
        print(f"Created Entry Group: {entry_group_id}")

    #Ensure Entry Type exists
    entry_type_id = "gcs-fileset-type"
    entry_type_path = client.entry_type_path(project_id, location, entry_type_id)
    try:
        client.get_entry_type(name=entry_type_path)
        print(f"Entry Type '{entry_type_id}' already exists.")
    except Exception:
        entry_type = dataplex_v1.EntryType()
        entry_type.display_name = "GCS Fileset Type" #could be JSON or any other types that you need to manage in your catalog
        entry_type.description = "Custom entry type for GCS filesets"
        operation = client.create_entry_type(
            parent=client.common_location_path(project_id, location),
            entry_type_id=entry_type_id,
            entry_type=entry_type,
        )
        operation.result()
        print(f"Created Entry Type: {entry_type_id}")

    # Recreate Entry
    entry_path = client.entry_path(project_id, location, entry_group_id, entry_id)
    try:
        client.delete_entry(name=entry_path)
    except Exception:
        pass

    entry = dataplex_v1.Entry()
    entry.entry_type = entry_type_path

    entry_source = dataplex_v1.EntrySource()
    entry_source.system = "gcs"
    entry_source.resource = gcs_path
    entry_source.display_name = display_name
    entry_source.description = f"GCS Fileset for user JSON data at {gcs_path}"
    entry.entry_source = entry_source

    created_entry = client.create_entry(
        parent=entry_group_path,
        entry_id=entry_id,
        entry=entry,
    )
    print(f"Registered GCS Entry: {created_entry.name}")

# Run GCS Registration
gcs_input_path = f"gs://{BUCKET_NAME}/input/users.json"
register_gcs_fileset(
    project_id=PROJECT_ID,
    location=REGION,
    entry_group_id=ENTRY_GROUP_ID,
    entry_id=ENTRY_ID,
    gcs_path=gcs_input_path,
    display_name="Users JSON GCS Fileset"
)

In [ ]:
# Create BigQuery table

from google.cloud import bigquery

def setup_bigquery_table(project_id, dataset_id, table_id, location):
    client = bigquery.Client(project=project_id)

    # Create dataset if not exists
    dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
    try:
        client.get_dataset(dataset_ref)
        print(f"BigQuery Dataset '{dataset_id}' already exists.")
    except Exception:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = location
        dataset.description = "Dataset for Spark Ingestion demo"
        client.create_dataset(dataset)
        print(f"Created BigQuery Dataset: {dataset_id}")

    # Create table if not exists
    table_ref = dataset_ref.table(table_id)
    schema = [
        bigquery.SchemaField("firstname", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("lastname", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("ipaddress", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("address", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("fullname", "STRING", mode="NULLABLE"),
    ]
    try:
        client.get_table(table_ref)
        print(f"BigQuery Table '{table_id}' already exists.")
    except Exception:
        table = bigquery.Table(table_ref, schema=schema)
        client.create_table(table)
        print(f"Created BigQuery Table: {table_id}")

setup_bigquery_table(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=REGION
)

In [ ]:
import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.4


In [ ]:
%pip install --quiet pyspark==3.5.4 google-cloud-datacatalog google-cloud-storage google-cloud-bigquery google-cloud-datacatalog-lineage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 4.2 MB/s eta 0:00:00


In [5]:
#Ingest the user.json data into the table and register the operation in lineage api

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit
from google.cloud import storage
from google.cloud import datacatalog_lineage_v1
import datetime
import os


#ENTRY_NAME = "projects/my-project/locations/us-central1/entryGroups/spark_demo_group/entries/gcs_users_json"

# 1. Redownload the file from GCS (just in case you don t execute the all the cells)
local_temp_file = "data/temp_users.json"
os.makedirs("data", exist_ok=True)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob("input/users.json")
blob.download_to_filename(local_temp_file)
print(f"Downloaded file locally: {local_temp_file}")

# Run Spark
spark = SparkSession.builder \
    .appName("Spark BigQuery Lineage Ingestion") \
    .config("spark.jars.packages", "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.35.0") \
    .getOrCreate()

try:
    # read file
    df = spark.read.json(local_temp_file)
    print("\nInput DataFrame schema:")
    df.printSchema()

    # do a light transform: combine first and last name
    print("Transforming: Combining 'firstname' and 'lastname' to 'fullname'...")
    transformed_df = df.withColumn(
        "fullname",
        concat(col("firstname"), lit(" "), col("lastname"))
    )
    print("\nTransformed DataFrame schema:")
    transformed_df.printSchema()

    # Ingest to BigQuery
    target_table = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    print(f"Writing transformed data to BigQuery table: {target_table}...")
    transformed_df.write \
        .format("bigquery") \
        .option("table", target_table) \
        .option("writeMethod", "direct") \
        .mode("append") \
        .save()
    print("Successfully wrote data to BigQuery!")

    # Confirm data ingestion
    print("\nVerifying BigQuery table contents:")
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=PROJECT_ID)
    query_job = bq_client.query(f"SELECT * FROM `{target_table}` LIMIT 5")
    results = query_job.result()
    for row in results:
        print(dict(row))

    # Record Ingestion Lineage using the Data Lineage API in Knowledge Catalog
    print(f"\n--- Recording Ingestion Lineage in Data Lineage API ---")
    lineage_client = datacatalog_lineage_v1.LineageClient()
    parent_path = f"projects/{PROJECT_ID}/locations/us-central1"

    # Create Process
    process = datacatalog_lineage_v1.Process()
    process.display_name = "Spark Ingestion Job"
    created_process = lineage_client.create_process(parent=parent_path, process=process)
    print(f"Process created/reused: {created_process.name}")

    # Create Run
    run = datacatalog_lineage_v1.Run()
    run.state = datacatalog_lineage_v1.Run.State.COMPLETED
    now = datetime.datetime.now(datetime.timezone.utc)
    run.start_time = now - datetime.timedelta(minutes=2)
    run.end_time = now
    created_run = lineage_client.create_run(parent=created_process.name, run=run)
    print(f"Run created: {created_run.name}")

    # Create Lineage Event linking GCS (physical source in GCS) -> Knowlege Catalog Entry (logical source) -> BigQuery Table (target)
    lineage_event = datacatalog_lineage_v1.LineageEvent()
    lineage_event.start_time = now - datetime.timedelta(minutes=2)
    lineage_event.end_time = now

    target_fqn = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{TABLE_ID}"

    # Link 1: Physical GCS -> Logical Knowlege Catalog Entry
    link1 = datacatalog_lineage_v1.EventLink()
    src1 = datacatalog_lineage_v1.EntityReference()
    src1.fully_qualified_name = f"gs://{BUCKET_NAME}/input/users.json"
    link1.source = src1

    tgt1 = datacatalog_lineage_v1.EntityReference()
    tgt1.fully_qualified_name = f"dataplex.googleapis.com/{ENTRY_NAME}"
    link1.target = tgt1

    # Link 2: Logical Knowlege Catalog Entry -> Target BigQuery Table
    link2 = datacatalog_lineage_v1.EventLink()
    src2 = datacatalog_lineage_v1.EntityReference()
    src2.fully_qualified_name = f"dataplex.googleapis.com/{ENTRY_NAME}"
    link2.source = src2

    tgt2 = datacatalog_lineage_v1.EntityReference()
    tgt2.fully_qualified_name = target_fqn
    link2.target = tgt2

    lineage_event.links.extend([link1, link2])

    created_event = lineage_client.create_lineage_event(parent=created_run.name, lineage_event=lineage_event)
    print(f"Recorded Lineage Event: {created_event.name}")

finally:
    spark.stop()
    if os.path.exists(local_temp_file):
        os.remove(local_temp_file)
        print("\nCleaned up temporary local JSON file.")

Downloaded GCS file locally to: data/temp_users.json

Input DataFrame schema:
root
 |-- address: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- ipaddress: string (nullable = true)
 |-- lastname: string (nullable = true)

Transforming: Combining 'firstname' and 'lastname' to 'fullname'...

Transformed DataFrame schema:
root
 |-- address: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- ipaddress: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- fullname: string (nullable = true)

Writing transformed data to BigQuery table.lineage_dataset.users_ingested_table...
Successfully wrote data to BigQuery!

Verifying BigQuery table contents:
{'firstname': 'Luffy', 'lastname': 'Monkey D.', 'ipaddress': '10.0.0.1', 'address': 'Foosha Village', 'fullname': 'Luffy Monkey D.'}
{'firstname': 'Zoro', 'lastname': 'Roronoa', 'ipaddress': '10.0.0.2', 'address': 'Shimotsuki Village', 'fullname': 'Zoro Roronoa'}
{'firstname': 'Nami', 'lastna